In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mlflow
import pandas as pd
import optuna

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils import resample

from chesswinnerprediction.baseline.double_stage_wrapper import Model
from chesswinnerprediction.baseline.custom_knn import CustomKNNClassifier
from chesswinnerprediction.baseline.utils import load_train_valid_test
from chesswinnerprediction.baseline.constants import BASELINE_RANDOM_STATE
from chesswinnerprediction.constants import WHITE_WIN_STR, BLACK_WIN_STR
from config import BASELINE_EXPERIMENT, MLRUNS_FOLDER_PATH

## MLFlow Configuration

In [3]:
mlflow.set_tracking_uri(MLRUNS_FOLDER_PATH)

In [4]:
if (experiment := mlflow.get_experiment_by_name(BASELINE_EXPERIMENT)) is not None:
    experiment_id = experiment.experiment_id
else:
    experiment_id = mlflow.create_experiment(BASELINE_EXPERIMENT)

In [5]:
mlflow.set_experiment(experiment_id=experiment_id)
# mlflow.sklearn.autolog(disable=True)


<Experiment: artifact_location='/home/tikhon/PycharmProjects/ChessWinnerPrediction/mlruns/299694667584362343', creation_time=1723328472086, experiment_id='299694667584362343', last_update_time=1723328472086, lifecycle_stage='active', name='baseline', tags={}>

## Load data

In [6]:
X_train, y_train, X_valid, y_valid, X_test, y_test = load_train_valid_test()

## Optuna

In [7]:
def balance_dataset(x, y, class_weight, random_state):
    data_df = pd.concat([x, y], axis=1)    
    base = y.value_counts().min() 
    
    balanced_df = pd.concat([
        resample(
            data_df[y == target],
            replace=(base * weight) > y.value_counts()[target],
            n_samples=int(base * weight),
            random_state=random_state
        )
        for target, weight in class_weight.items()
    ])
    
    return balanced_df[x.columns], balanced_df[y.name]

In [30]:
def get_splitter_model(trial: optuna.Trial, model_type, splitter_type):
    if model_type == "decision_tree":
        prefix = f"{splitter_type}-{model_type}-"
        model = DecisionTreeClassifier(
            random_state=BASELINE_RANDOM_STATE,
            class_weight="balanced",
            max_depth=trial.suggest_int(f"{prefix}max_depth", 2, 32),
            min_samples_leaf=trial.suggest_int(f"{prefix}min_samples_leaf", 1, X_train.shape[0]),
            max_features=trial.suggest_categorical(f"{prefix}max_features", [None, "sqrt", "log2"]),
        )
    elif model_type == "random_forest":
        prefix = f"{splitter_type}-{model_type}-"
        model = RandomForestClassifier(
            random_state=BASELINE_RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
            n_estimators=trial.suggest_int(f"{prefix}n_estimators", 10, 800),
            max_depth=trial.suggest_int(f"{prefix}max_depth", 2, 20),
            min_samples_leaf=trial.suggest_int(f"{prefix}min_samples_leaf", 1, X_train.shape[0]),
            max_features=trial.suggest_categorical(f"{prefix}max_features", [None, "sqrt", "log2"]),
        )
    elif model_type == "gradient_boosting":
        prefix = f"{splitter_type}-{model_type}-"
        model = GradientBoostingClassifier(
            random_state=BASELINE_RANDOM_STATE,
            n_estimators=trial.suggest_int(f"{prefix}n_estimators", 2, 300),
            max_depth=trial.suggest_int(f"{prefix}max_depth", 2, 16),
            min_samples_leaf=trial.suggest_int(f"{prefix}min_samples_leaf", 1, X_train.shape[0]),
            max_features=trial.suggest_categorical(f"{prefix}max_features", [None, "sqrt", "log2"]),
        )
    elif model_type == "logistic_regression":
        prefix = f"{splitter_type}-{model_type}-"
        model = LogisticRegression(
            random_state=BASELINE_RANDOM_STATE,
            class_weight="balanced",
            solver="liblinear",
            penalty="l2",
            max_iter=100,
            C=trial.suggest_float(f"{prefix}C", 0.01, 10),
            tol=trial.suggest_float(f"{prefix}tol", 0.001, 1)
        )
    elif model_type == "custom_knn":
        prefix = f"{splitter_type}-{model_type}-"
        
        if splitter_type == "win_to_draw":
            class_weights = {
                Model.win_to_draw_splitter_win_symbol: trial.suggest_float(f"{prefix}win_constant", 1, 1),
                Model.win_to_draw_splitter_draw_symbol: trial.suggest_float(f"{prefix}draw_constant", 1, 1),
            }
        else:
            black_white_win_constant = trial.suggest_float(f"{prefix}black_white_win_constant", 1, 1)
            class_weights = {
                WHITE_WIN_STR: black_white_win_constant,
                BLACK_WIN_STR: black_white_win_constant,
            }
        model = CustomKNNClassifier(
            random_state=BASELINE_RANDOM_STATE,
            class_weight=class_weights,
            balance_dataset_func=balance_dataset,
            score_size=0.25,
            n_jobs=-1,
            weights=trial.suggest_categorical(f"{prefix}weights", ["uniform", "distance"]),
            n_neighbors=trial.suggest_int(f"{prefix}n_neighbors", 2, 100),
            leaf_size=trial.suggest_int(f"{prefix}leaf_size", 10, 100),
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    return model

## Optuna Optimization

In [35]:
class DoubleStageOptunaTuner:
    def __init__(self, get_splitter_model_func=get_splitter_model):
        self.get_splitter_model_func = get_splitter_model_func
        
        self.model_name_to_run_id = {}
        self.model_suggestion_list = ["decision_tree", "random_forest", "gradient_boosting", "logistic_regression", "custom_knn"]
        self.models = [DecisionTreeClassifier, RandomForestClassifier, GradientBoostingClassifier, LogisticRegression, CustomKNNClassifier]
        self.models_dict = dict(zip(self.model_suggestion_list, self.models))
        
        self.study = None
    
    @staticmethod
    def get_model_name(win_to_draw_splitter_type, black_to_white_splitter_type):
        return f"{win_to_draw_splitter_type}-{black_to_white_splitter_type}"
        
    def get_model(self, trial):
        win_to_draw_splitter_type = trial.suggest_categorical("win_to_draw_splitter_type", self.model_suggestion_list)
        black_to_white_splitter_type = trial.suggest_categorical("black_to_white_splitter_type", self.model_suggestion_list)
        
        win_to_draw_model = self.get_splitter_model_func(trial, win_to_draw_splitter_type, "win_to_draw")
        black_to_white_model = self.get_splitter_model_func(trial, black_to_white_splitter_type, "black_to_white")
        
        model_name = self.get_model_name(win_to_draw_splitter_type, black_to_white_splitter_type)
        return Model(win_to_draw_model, black_to_white_model, model_name)
        
    
    def objective(self, trial: optuna.Trial):
        model = self.get_model(trial)
        model.fit(X_train, y_train)
        
        score = model.score(X_valid, y_valid)
        
        with mlflow.start_run(nested=True, run_id=self.model_name_to_run_id[model.name]):
            with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as trial_run:
                mlflow.log_metric("score", score, run_id=trial_run.info.run_id)
                
                mlflow.log_metric("score", score, run_id=trial_run.info.run_id)
                mlflow.log_params(trial.params, run_id=trial_run.info.run_id)
                
        return score
    
    def generate_mode_name_to_run_id_dict(self):
        self.model_name_to_run_id.clear()
        for model_1 in self.model_suggestion_list:
            for model_2 in self.model_suggestion_list:
                model_name = f"{model_1}-{model_2}"
                with mlflow.start_run(nested=True, run_name=model_name) as run:
                    self.model_name_to_run_id[model_name] = run.info.run_id
    
    def run(self, n_trials=10, show_progress_bar=True, n_jobs=-1):
        with mlflow.start_run(run_name="Optuna"):
            self.generate_mode_name_to_run_id_dict()
            
            self.study = optuna.create_study(direction="maximize", study_name="double_stage_log_reg")
            self.study.optimize(self.objective, n_trials=n_trials, show_progress_bar=show_progress_bar, n_jobs=n_jobs)

            mlflow.log_params(self.study.best_params)
            mlflow.log_metric("best_value", self.study.best_value)
        
        return self.study

In [36]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [37]:
tuner = DoubleStageOptunaTuner(get_splitter_model)
study = tuner.run(n_trials=128, show_progress_bar=True, n_jobs=-1)

  0%|          | 0/128 [00:00<?, ?it/s]

In [38]:
study.best_params

{'win_to_draw_splitter_type': 'gradient_boosting',
 'black_to_white_splitter_type': 'logistic_regression',
 'win_to_draw-gradient_boosting-n_estimators': 284,
 'win_to_draw-gradient_boosting-max_depth': 2,
 'win_to_draw-gradient_boosting-min_samples_leaf': 4949,
 'win_to_draw-gradient_boosting-max_features': None,
 'black_to_white-logistic_regression-C': 5.040817135091013,
 'black_to_white-logistic_regression-tol': 0.8284562928765168}

## Best Model